# Week 10A — Motion Analysis and Optical Flow

**MCTA 4364 Machine Vision** · Session 1 of Week 10 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W10_motion_tracking/W10A_motion_optical_flow.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W10_motion_tracking/W10A_motion_optical_flow.ipynb)

So far every image was analysed on its own. A camera on a production line, a robot or a vehicle delivers **video**: the same scene, a few
milliseconds apart. The differences between frames tell us what moves, where and how fast. This session covers the two classical families:
**change detection** (frame differencing and background subtraction: *what* moves) and **optical flow** (*how* each pixel moves), measured against
ground truth on a simulated conveyor.

### Learning outcomes
By the end of this session you will be able to:
1. Detect moving objects with frame differencing and background models (running average, median, MOG2, KNN), and evaluate them with F1.
2. Derive the **optical-flow constraint** from brightness constancy and explain the **aperture problem**.
3. Solve **Lucas–Kanade** as a least-squares problem and connect it to the structure tensor of Week 5.
4. Compute sparse (pyramidal LK) and dense (Farnebäck, DIS) flow, visualise it, and measure its **end-point error**.
5. Convert flow into physical speed (mm/s) with the camera calibration.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1 | Video as a 3D signal; our two test videos |
| 8–30 | 2–3 | Frame differencing; background subtraction (interactive); evaluation |
| 30–50 | 4–5 | Optical-flow constraint, aperture problem, Lucas–Kanade |
| 50–68 | 6–7 | Sparse and dense flow; accuracy vs ground truth; speed in mm/s |
| 68–80 | 10 | Quiz and exit ticket |
| Home | 8–9 | Exercises with self-checks, challenge |

**Hardware:** a laptop CPU is enough (about 1 minute). No downloads except OpenCV sample videos (a few MB).

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import partsim
from cvhelpers import SMOKE, check, interact, load_sample, opencv_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
N_FRAMES = 40 if SMOKE else 150
BELT_SPEED = 6.0                                        # px per frame, the ground truth of the simulator
frames, gt = partsim.make_conveyor_video(N_FRAMES, seed=1, speed=BELT_SPEED)
gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi")))
street = []
while len(street) < (60 if SMOKE else 200):
    ok, f = cap.read()
    if not ok:
        break
    street.append(f)
cap.release()
street_gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in street]
print(f"conveyor: {len(frames)} frames of {frames[0].shape[1]} x {frames[0].shape[0]} | vtest.avi: {len(street)} frames, 10 FPS")

## 1. Video as a function of space **and time**

A video is $I(x, y, t)$. Stacking frames gives a 3D volume; a moving object draws a slanted "tube" through it. Two useful views:
- a **frame** $I(\cdot, \cdot, t)$: what you normally see;
- a **space–time slice** $I(x, y_0, \cdot)$: one image row over time. Objects moving at constant speed become **straight lines** whose slope is their velocity.

In [ ]:
y0 = int(gt[min(60, N_FRAMES - 1)]["boxes"][0][1] + 10) if len(gt[min(60, N_FRAMES - 1)]["boxes"]) else 180
slice_conv = np.stack([g[y0] for g in gray])
rows = np.stack(street_gray).astype(np.float32).std(0).mean(1)          # pick the street row where most happens
y_st = int(np.argmax(rows[150:500]) + 150)
slice_street = np.stack([g[y_st] for g in street_gray])
fig, axes = plt.subplots(2, 2, figsize=(20, 8.5), gridspec_kw=dict(height_ratios=[1, 1.1]))
axes[0, 0].imshow(frames[min(60, N_FRAMES - 1)][..., ::-1]); axes[0, 0].axhline(y0, color="#fbbc05"); axes[0, 0].set_title(f"conveyor frame 60, row y = {y0} (yellow)")
axes[0, 1].imshow(street[min(60, len(street) - 1)][..., ::-1]); axes[0, 1].axhline(y_st, color="#fbbc05"); axes[0, 1].set_title(f"vtest.avi frame 60, row y = {y_st}")
axes[1, 0].imshow(slice_conv, cmap="gray", aspect="auto"); axes[1, 0].set_xlabel("x (px)"); axes[1, 0].set_ylabel("frame t")
axes[1, 0].set_title(f"space-time slice: parallel lines, slope = belt speed ({BELT_SPEED:.0f} px/frame)")
axes[1, 1].imshow(slice_street, cmap="gray", aspect="auto"); axes[1, 1].set_xlabel("x (px)"); axes[1, 1].set_ylabel("frame t")
axes[1, 1].set_title("pedestrians: different slopes = different speeds and directions")
for ax in axes[0]:
    ax.axis("off")
plt.suptitle("Figure 1.1 - Motion becomes geometry in the space-time volume", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Change detection

**Frame differencing**: $D_t = |I_t - I_{t-1}| > \tau$. Cheap, but it only responds where intensity *changes*: the leading and trailing edges of an
object, not its uniform interior, and nothing at all for an object that stops.

**Background subtraction** keeps a model $B$ of the empty scene and marks $|I_t - B| > \tau$ as foreground:

| model | update | notes |
|---|---|---|
| running average | $B \leftarrow (1-\alpha) B + \alpha I_t$ | adapts to slow lighting changes; ghosts when objects stop |
| temporal median | median of the last $N$ frames | robust if objects occupy a pixel < 50% of the time |
| **MOG2** (Zivkovic, 2004) | a mixture of Gaussians per pixel, adaptive number of components | handles flicker, waving trees; detects shadows |
| **KNN** | per-pixel sample set, K-nearest-neighbour test | strong on dynamic backgrounds |

We compare them to the simulator's **ground-truth foreground mask** with the F1 score $= 2|M \cap G| / (|M| + |G|)$.

In [ ]:
def f1(mask, truth):
    tp = np.logical_and(mask, truth).sum()
    return 2 * tp / (mask.sum() + truth.sum() + 1e-9)


kernel = np.ones((3, 3), np.uint8)
tidy = lambda m: cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, kernel) > 0
WARMUP = N_FRAMES // 3
masks = {k: [] for k in ["frame difference", "running average", "temporal median", "MOG2", "KNN"]}
bg_avg = gray[0].astype(np.float32)
median_bg = np.median(np.stack(gray[:WARMUP]), 0)
mog, knn = cv2.createBackgroundSubtractorMOG2(200, 16, detectShadows=False), cv2.createBackgroundSubtractorKNN(200, 400, detectShadows=False)
for t in range(N_FRAMES):
    m_mog, m_knn = mog.apply(frames[t]) > 0, knn.apply(frames[t]) > 0
    if t >= WARMUP:
        masks["frame difference"].append(tidy(cv2.absdiff(gray[t], gray[t - 1]) > 25))
        masks["running average"].append(tidy(np.abs(gray[t] - bg_avg) > 25))
        masks["temporal median"].append(tidy(np.abs(gray[t].astype(np.float32) - median_bg) > 25))
        masks["MOG2"].append(tidy(m_mog)); masks["KNN"].append(tidy(m_knn))
    if t > 0:
        cv2.accumulateWeighted(gray[t], bg_avg, 0.02)
scores = {k: np.mean([f1(m, gt[WARMUP + i]["fg"]) for i, m in enumerate(v)]) for k, v in masks.items()}
t_show = N_FRAMES - 10
fig, axes = plt.subplots(2, 4, figsize=(22, 7.5))
axes[0, 0].imshow(frames[t_show][..., ::-1]); axes[0, 0].set_title(f"frame {t_show}")
axes[0, 1].imshow(gt[t_show]["fg"], cmap="gray"); axes[0, 1].set_title("ground-truth foreground")
axes[0, 2].imshow(median_bg, cmap="gray"); axes[0, 2].set_title("median background model")
axes[0, 3].bar(list(scores), list(scores.values()), color=["#d93025", "#fbbc05", "#188038", "#1a73e8", "#9334e6"])
axes[0, 3].set_ylim(0, 1); axes[0, 3].set_title("mean F1 against ground truth"); axes[0, 3].tick_params(axis="x", rotation=30)
for ax, (k, v) in zip(axes[1], list(masks.items())[:1] + list(masks.items())[2:]):
    ax.imshow(v[t_show - WARMUP], cmap="gray"); ax.set_title(f"{k}: F1 {scores[k]:.2f}")
for ax in list(axes[0, :3]) + list(axes[1]):
    ax.axis("off")
plt.suptitle("Figure 2.1 - Change detection on the conveyor: frame differencing only sees edges", fontweight="bold")
plt.tight_layout(); plt.show()
for k, v in scores.items():
    print(f"{k:<18} F1 = {v:.3f}")

### 🎛️ Interactive: MOG2 on a real street
Vary the learning history and the variance threshold. Watch shadows, the people who stand still, and noise.

In [ ]:
def mog_demo(history=200, var_threshold=16, detect_shadows=True, frame=150):
    frame = min(frame, len(street) - 1)
    sub = cv2.createBackgroundSubtractorMOG2(history, var_threshold, detectShadows=detect_shadows)
    for f in street[:frame + 1]:
        fg = sub.apply(f)
    vis = np.zeros_like(street[frame]); vis[fg == 255] = (255, 255, 255); vis[fg == 127] = (0, 140, 255)
    show(street[frame], vis, sub.getBackgroundImage(), titles=[f"frame {frame}", "foreground (orange = detected shadow)", "learned background"],
         figsize=(20, 5))


interact(mog_demo, history=widgets.IntSlider(value=200, min=10, max=500, step=10), var_threshold=widgets.IntSlider(value=16, min=4, max=100),
         detect_shadows=True, frame=widgets.IntSlider(value=150, min=10, max=199))

## 3. From foreground to objects

A foreground mask is not yet a list of objects. The classical chain is: **clean** the mask (morphology, Week 3B) → **connected components**
(Week 4B) → keep blobs larger than a minimum area → bounding boxes. Touching or overlapping objects merge into one blob — one of the reasons learned
detectors (Week 8) replaced this pipeline for crowded scenes. A static camera and background subtraction remain excellent for **uncluttered**
scenes and as a cheap *motion trigger* ("only run the expensive detector when something moves").

In [ ]:
m = cv2.morphologyEx(masks["temporal median"][t_show - WARMUP].astype(np.uint8) * 255, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
n, lab, stats, cent = cv2.connectedComponentsWithStats(m)
blobs = [s for s in stats[1:] if s[4] > 150]
vis = frames[t_show].copy()
for x, y, w, h, a in blobs:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 255), 2)
show(vis, partsim.draw_boxes(frames[t_show], gt[t_show]["boxes"], gt[t_show]["labels"]),
     titles=[f"{len(blobs)} blobs from background subtraction", f"{len(gt[t_show]['boxes'])} ground-truth objects"], figsize=(20, 5.5))

## 4. Optical flow: the constraint equation and the aperture problem

**Brightness constancy**: a point keeps its intensity as it moves by $(u, v)$ in one frame, $I(x+u, y+v, t+1) = I(x, y, t)$. A first-order Taylor
expansion gives the **optical-flow constraint**:

$$I_x u + I_y v + I_t = 0.$$

One equation, two unknowns: at a single pixel only the flow component **along the gradient** (the *normal flow*) can be measured. This is the
**aperture problem**: a straight edge seen through a small window looks the same whatever its motion along the edge. Corners (two gradient
directions) resolve it — the same argument as for Harris corners in Week 5A.

In [ ]:
AXIS = np.array([1.0, 1.0]) / np.sqrt(2)            # the bar runs down-right on screen
NORMAL = np.array([1.0, -1.0]) / np.sqrt(2)         # perpendicular to the bar


def bar_image(shift, size=120, half_width=9.0):
    """A long, slightly blurred bar through the centre, translated by `shift` (x, y) in pixels (sub-pixel accurate)."""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    s = (xx - size / 2 - shift[0]) * NORMAL[0] + (yy - size / 2 - shift[1]) * NORMAL[1]
    return 255 / (1 + np.exp(-(half_width - np.abs(s)) / 1.5))


yy_, xx_ = np.mgrid[0:120, 0:120]
ap = np.hypot(xx_ - 60, yy_ - 60) < 30
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for r, (d, name) in enumerate([(np.array([1.0, 0.0]), "moving right"), (AXIS.copy(), "moving along the bar")]):
    for k, t in enumerate([-8, 0, 8]):
        axes[r, k].imshow(np.where(ap, bar_image(d * t), 40), cmap="gray", vmin=0, vmax=255)
        axes[r, k].set_title(f"{name}, t = {t}"); axes[r, k].axis("off")
    i0, i1 = bar_image(np.zeros(2)), bar_image(0.5 * d)             # a half-pixel step, so the Taylor expansion holds
    gy, gx = np.gradient(i0)
    it = i1 - i0
    sel = ap & (np.hypot(gx, gy) > 2)
    g2 = gx[sel] ** 2 + gy[sel] ** 2
    axes[r, 3].scatter(-it[sel] * gx[sel] / g2, -it[sel] * gy[sel] / g2, s=6, alpha=0.5, label="normal flow at each pixel")
    axes[r, 3].plot([0, 0.5 * d[0]], [0, 0.5 * d[1]], "r-", lw=2.5, label="true motion")
    axes[r, 3].set_xlim(-0.7, 0.7); axes[r, 3].set_ylim(0.7, -0.7); axes[r, 3].set_aspect("equal"); axes[r, 3].grid(alpha=0.3)
    axes[r, 3].legend(fontsize=8, loc="lower left"); axes[r, 3].set_xlabel("u (px)"); axes[r, 3].set_ylabel("v (px)")
    axes[r, 3].set_title("measured vs true motion (one half-pixel step)")
plt.suptitle("Figure 4.1 - The aperture problem: only the motion component perpendicular to the edge is measured; motion along the bar is invisible", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Lucas–Kanade

Assume the flow is **constant in a small window** (e.g. 15 × 15). Every pixel $i$ of the window gives one constraint, so we get an over-determined
system $A\mathbf{d} = \mathbf{b}$ with $A = [I_{x,i}\;\; I_{y,i}]$, $\mathbf{b} = [-I_{t,i}]$. Least squares:

$$\underbrace{\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}}_{\text{structure tensor } M}
\begin{bmatrix} u \\ v \end{bmatrix} = -\begin{bmatrix} \sum I_x I_t \\ \sum I_y I_t \end{bmatrix}.$$

$M$ is invertible and well-conditioned exactly where **both eigenvalues are large**: at corners (Shi–Tomasi called them *good features to track*).
The Taylor expansion only holds for motions of about a pixel, so real implementations are **pyramidal** (Week 5A): estimate the motion at a coarse
level, warp, and refine at each finer level; a few **iterations** per level handle the non-linearity.

In [ ]:
t0 = min(80, N_FRAMES - 4)
a, b = gray[t0], gray[t0 + 1]
p0 = cv2.goodFeaturesToTrack(a, 300, 0.01, 6, mask=gt[t0]["fg"].astype(np.uint8) * 255)
GAP = 3                                                    # compare frames 3 apart: an 18 px motion, larger than the 15 px window
res_lk = {}
for levels in [0, 1, 3]:
    p1, st, err = cv2.calcOpticalFlowPyrLK(a, gray[t0 + GAP], p0, None, winSize=(15, 15), maxLevel=levels)
    d = (p1 - p0).reshape(-1, 2)
    e = np.hypot(d[:, 0] - GAP * BELT_SPEED, d[:, 1])
    res_lk[levels] = (e, np.mean((st.ravel() == 1) & (e < 1.0)))
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8), gridspec_kw=dict(width_ratios=[1.5, 1]))
vis = frames[t0].copy()
p1, st, _ = cv2.calcOpticalFlowPyrLK(a, b, p0, None, winSize=(15, 15), maxLevel=3)
for q0, q1, s in zip(p0.reshape(-1, 2), p1.reshape(-1, 2), st.ravel()):
    if s:
        cv2.arrowedLine(vis, tuple(q0.astype(int)), tuple((q0 + 4 * (q1 - q0)).astype(int)), (0, 255, 0), 1, tipLength=0.3)
axes[0].imshow(vis[..., ::-1]); axes[0].axis("off"); axes[0].set_title("pyramidal LK on corners of the parts (arrows x4)")
axes[1].boxplot([v[0] for v in res_lk.values()])                       # tick labels set separately: works with old and new matplotlib
axes[1].set_xticks(range(1, len(res_lk) + 1)); axes[1].set_xticklabels([f"{k} pyramid levels\n({v[1]:.0%} within 1 px)" for k, v in res_lk.items()])
axes[1].set_ylabel("end-point error (px)"); axes[1].set_yscale("log"); axes[1].grid(alpha=0.3)
axes[1].set_title(f"a {GAP * BELT_SPEED:.0f} px motion (frames {GAP} apart) needs a pyramid")
plt.suptitle("Figure 5.1 - Sparse Lucas-Kanade flow and the effect of the pyramid", fontweight="bold")
plt.tight_layout(); plt.show()
for k, (e, s) in res_lk.items():
    print(f"maxLevel {k}: median end-point error {np.median(e):.2f} px, {s:.0%} of points within 1 px of the truth")

In [ ]:
# Sparse tracking over time on the street video: corner trails
p = cv2.goodFeaturesToTrack(street_gray[0], 150, 0.01, 8)
trails = [p.reshape(-1, 2)]
alive = np.ones(len(p), bool)
for t in range(1, min(40, len(street_gray))):
    p_new, st, _ = cv2.calcOpticalFlowPyrLK(street_gray[t - 1], street_gray[t], p, None, winSize=(15, 15), maxLevel=3)
    alive &= st.ravel() == 1
    p = p_new; trails.append(p.reshape(-1, 2))
trails = np.stack(trails)
vis = street[len(trails) - 1].copy()
moved = np.hypot(*(trails[-1] - trails[0]).T) > 5
for i in np.nonzero(alive & moved)[0]:
    cv2.polylines(vis, [trails[:, i].astype(np.int32)], False, (0, 255, 255), 2)
show(vis, titles=[f"LK trails over {len(trails)} frames: {int((alive & moved).sum())} moving corners (static corners hidden)"], figsize=(11, 8))

## 6. Dense optical flow

Dense methods estimate $(u, v)$ at **every pixel**:
- **Farnebäck** (2003): fits a quadratic polynomial to each neighbourhood and derives the displacement from how the polynomial shifts.
- **DIS** (Dense Inverse Search, Kroeger et al., 2016): fast patch matching on a pyramid plus variational refinement; ~100+ FPS on a CPU.
- **Horn–Schunck** (1981), the classical variational method: brightness constancy + a global smoothness term.
- **Deep flow**: FlowNet, PWC-Net, **RAFT** (2020) and successors (SEA-RAFT, 2024) learn the matching; they dominate the benchmarks (Sintel, KITTI).

Flow is shown with a colour wheel: **hue = direction, brightness = speed**.

In [ ]:
def flow_to_rgb(flow, max_mag=None):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros(flow.shape[:2] + (3,), np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = np.clip(mag / (max_mag or mag.max() + 1e-9) * 255, 0, 255)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


fb = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.2, 0)
dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
fd = dis.calc(a, b, None)
fg = gt[t0]["fg"]
epe = lambda f: np.hypot(f[..., 0] - BELT_SPEED, f[..., 1])[fg]
wheel_y, wheel_x = np.mgrid[-1:1:101j, -1:1:101j]
wheel = flow_to_rgb(np.dstack([wheel_x, wheel_y]).astype(np.float32), 1.0) * (np.hypot(wheel_x, wheel_y) <= 1)[..., None]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.2), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.35]))
axes[0].imshow(frames[t0][..., ::-1]); axes[0].set_title(f"frame {t0}")
axes[1].imshow(flow_to_rgb(fb, 2 * BELT_SPEED)); axes[1].set_title(f"Farneback: EPE on parts {epe(fb).mean():.2f} px")
axes[2].imshow(flow_to_rgb(fd, 2 * BELT_SPEED)); axes[2].set_title(f"DIS: EPE on parts {epe(fd).mean():.2f} px")
axes[3].imshow(wheel); axes[3].set_title("colour key")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Figure 6.1 - Dense flow on the conveyor (true motion: {BELT_SPEED:.0f} px/frame to the right = red)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
r1, r2 = load_sample("rubberwhale1.png"), load_sample("rubberwhale2.png")
g1, g2 = cv2.cvtColor(r1, cv2.COLOR_BGR2GRAY), cv2.cvtColor(r2, cv2.COLOR_BGR2GRAY)
t_ = {}
t1 = time.perf_counter(); f_fb = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0); t_["Farneback"] = time.perf_counter() - t1
t1 = time.perf_counter(); f_dis = dis.calc(g1, g2, None); t_["DIS"] = time.perf_counter() - t1
# check: warp frame 2 back with the flow; a good flow makes it look like frame 1
hh, ww = g1.shape
grid = np.dstack(np.meshgrid(np.arange(ww), np.arange(hh))).astype(np.float32)
warped = cv2.remap(r2, grid + f_dis, None, cv2.INTER_LINEAR)
fig, axes = plt.subplots(1, 5, figsize=(24, 4.2))
axes[0].imshow(r1[..., ::-1]); axes[0].set_title("rubberwhale1 (Middlebury)")
axes[1].imshow(flow_to_rgb(f_fb, 3)); axes[1].set_title(f"Farneback ({t_['Farneback'] * 1000:.0f} ms)")
axes[2].imshow(flow_to_rgb(f_dis, 3)); axes[2].set_title(f"DIS ({t_['DIS'] * 1000:.0f} ms)")
axes[3].imshow(np.abs(r2.astype(int) - r1.astype(int)).mean(2), cmap="magma", vmin=0, vmax=40); axes[3].set_title("|frame 2 - frame 1|")
axes[4].imshow(np.abs(warped.astype(int) - r1.astype(int)).mean(2), cmap="magma", vmin=0, vmax=40); axes[4].set_title("|frame 2 warped by DIS flow - frame 1|")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 6.2 - Dense flow on a real sequence, checked by warping (a good flow cancels the difference)", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. From pixels per frame to millimetres per second

With the calibration of Week 2 (here: 0.5 mm/px on the belt plane) and the frame rate, flow becomes a physical speed:

$$v = \lVert (u, v) \rVert \;\times\; \text{mm/px} \;\times\; \text{FPS}.$$

A robust estimate uses the **median** over all foreground pixels (outliers at object borders and occlusions are common). This is how a camera can
replace an encoder to measure belt speed, or detect a slipping belt.

In [ ]:
FPS = 30.0
speeds = []
for t in range(WARMUP, N_FRAMES - 1):
    f = dis.calc(gray[t], gray[t + 1], None)
    m = masks["temporal median"][t - WARMUP]
    if m.sum() > 50:
        speeds.append(np.median(np.hypot(f[..., 0], f[..., 1])[m]) * partsim.SCENE_MM_PER_PX * FPS)
true_speed = BELT_SPEED * partsim.SCENE_MM_PER_PX * FPS
plt.figure(figsize=(12, 3.8))
plt.plot(np.arange(WARMUP, WARMUP + len(speeds)), speeds, "o-", ms=3, label="median DIS flow on the foreground")
plt.axhline(true_speed, color="k", ls="--", label=f"true belt speed {true_speed:.0f} mm/s")
plt.xlabel("frame"); plt.ylabel("mm/s"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"Figure 7.1 - Belt speed from optical flow: {np.median(speeds):.1f} mm/s measured (median), {true_speed:.0f} mm/s true", fontweight="bold")
plt.show()

## 8. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Three-frame differencing
Frame differencing marks both the old and the new position of an object. The **three-frame** version keeps only pixels that changed in *both* $|I_t - I_{t-1}|$ and $|I_{t+1} - I_t|$ (above `thr`), i.e. the object at time $t$. Return a boolean mask.

In [ ]:
def three_frame_diff(prev, curr, nxt, thr=25):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    z = np.zeros((20, 40), np.uint8)
    f0, f1_, f2 = z.copy(), z.copy(), z.copy()
    f0[5:15, 2:8] = 200; f1_[5:15, 12:18] = 200; f2[5:15, 22:28] = 200
    m = np.asarray(three_frame_diff(f0, f1_, f2), bool)
    assert m.shape == z.shape and m.dtype == bool, "return a boolean mask of the image size"
    assert m[5:15, 12:18].all() and m.sum() == 60, f"only the object at time t should be marked, got {m.sum()} pixels"
    t = min(60, N_FRAMES - 2)
    m2 = np.asarray(three_frame_diff(gray[t - 1], gray[t], gray[t + 1]), bool)
    near = cv2.dilate(gt[t]["fg"].astype(np.uint8), np.ones((15, 15), np.uint8)) > 0    # parts and their moving shadows
    assert (m2 & ~near).sum() < 0.2 * max(m2.sum(), 1), "on the conveyor, most marked pixels should be on or next to the parts"


check("E1 three_frame_diff", _test_e1)

### E2. Lucas–Kanade at one point
Estimate the flow `(u, v)` at pixel `(x, y)` between float images `I0` and `I1` with a single (non-pyramidal) LK step: use central-difference gradients of `I0` (`np.gradient`), `It = I1 - I0`, a `(2*half+1)`² window, and solve the 2 × 2 system of Section 5.

In [ ]:
def lk_point(I0, I1, x, y, half=7):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    yy, xx = np.mgrid[0:64, 0:64].astype(np.float32)
    blob = lambda cx, cy: 100 * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 6.0 ** 2))
    u, v = np.asarray(lk_point(blob(32, 32), blob(32.4, 31.7), 32, 32), float)
    assert abs(u - 0.4) < 0.08 and abs(v + 0.3) < 0.08, f"expected about (0.4, -0.3), got ({u:.3f}, {v:.3f})"
    u2, v2 = np.asarray(lk_point(blob(32, 32), blob(31.2, 32.5), 30, 34), float)
    assert abs(u2 + 0.8) < 0.15 and abs(v2 - 0.5) < 0.15, f"expected about (-0.8, 0.5), got ({u2:.3f}, {v2:.3f})"


check("E2 lk_point", _test_e2)

### E3. End-point error
Return the **mean** end-point error $\lVert \mathbf{f} - \mathbf{f}_{gt} \rVert$ between two flow fields `[H, W, 2]`, over the pixels where `mask` is True (all pixels if `mask` is None).

In [ ]:
def endpoint_error(flow, flow_gt, mask=None):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    f = np.zeros((4, 5, 2), np.float32); g = np.zeros_like(f); g[..., 0] = 3; g[..., 1] = 4
    assert abs(endpoint_error(f, g) - 5) < 1e-6, "3-4-5 triangle: EPE should be 5"
    m = np.zeros((4, 5), bool); m[0, 0] = True; f[0, 0] = [3, 4]
    assert abs(endpoint_error(f, g, m)) < 1e-6, "only masked pixels count"
    gt_flow = np.zeros_like(fd); gt_flow[..., 0] = BELT_SPEED
    assert abs(endpoint_error(fd, gt_flow, fg) - epe(fd).mean()) < 1e-4, "differs from Figure 6.1"


check("E3 endpoint_error", _test_e3)

### E4. When does brightness constancy fail? (no self-check)
Regenerate the conveyor video with `flicker=0.08` (a flickering lamp) and with `noise=10`. How do the F1 of MOG2 / median and the EPE of DIS change?
Which assumption of each method is violated? Propose one fix for each (pre-processing or model choice).

In [ ]:
# E4: your code here

## 9. Challenge: a slipping-belt alarm
Simulate a belt that slows from 6 to 3 px/frame between frames 60 and 90 (modify the object speeds inside a copy of `make_conveyor_video`). Build a
monitor that estimates the speed every frame from DIS flow, smooths it (moving median), and raises an alarm when it deviates by more than 10% from
the nominal speed for more than 5 frames. Report the detection delay and the false-alarm rate on a normal video.

In [ ]:
# Your challenge code here

## 10. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why does frame differencing miss the interior of a uniformly coloured moving object?',
        "options": [
            'The threshold is too low',
            'Inside the object the intensity does not change between frames',
            'It only works in colour',
            'Because of the pyramid',
        ],
        "answer": 'a55672555d',
        "explain": 'Only the leading and trailing edges change.',
    },
    {
        "q": 'What is the optical-flow constraint equation?',
        "options": [
            'I_x u + I_y v + I_t = 0',
            'u + v = 0',
            'I_x + I_y = I_t',
            'u = I_x / I_t',
        ],
        "answer": 'f7de071f7f',
        "explain": 'From brightness constancy and a first-order Taylor expansion.',
    },
    {
        "q": 'What does the aperture problem say?',
        "options": [
            'Lenses have finite apertures',
            'Along a straight edge only the motion perpendicular to the edge can be measured locally',
            'Flow is undefined at corners',
            'Large motions cannot be measured',
        ],
        "answer": 'd73d8a51fc',
        "explain": 'One equation, two unknowns.',
    },
    {
        "q": 'Lucas-Kanade is well-conditioned where...',
        "options": [
            'the image is flat',
            'the structure tensor has two large eigenvalues (corners)',
            'there is one strong edge',
            'the image is dark',
        ],
        "answer": '51ed4341db',
        "explain": "Same condition as Shi-Tomasi 'good features to track'.",
    },
    {
        "q": 'Why is Lucas-Kanade implemented on a pyramid?',
        "options": [
            'To save memory',
            'The linearisation only holds for small motions; coarse levels make large motions small',
            'To detect colours',
            'To remove noise',
        ],
        "answer": '0c7b405de9',
        "explain": 'Estimate coarse, then refine.',
    },
    {
        "q": 'A part moves 4 px/frame at 0.5 mm/px and 25 FPS. Its speed?',
        "options": [
            '2 mm/s',
            '50 mm/s',
            '100 mm/s',
            '200 mm/s',
        ],
        "answer": '579e2c7573',
        "explain": '4 x 0.5 x 25 = 50 mm/s.',
    },
    {
        "q": 'Which background model best tolerates objects that pass through a pixel less than half of the time?',
        "options": [
            'Frame differencing',
            'Temporal median',
            'The first frame',
            'Running maximum',
        ],
        "answer": 'db1da443fb',
        "explain": 'The median ignores the minority of samples covered by objects.',
    },
], title='Week 10A quiz')

## 11. Exit ticket (reflection)
1. Your camera is mounted on a moving robot arm. Which of today's methods still work, and which break? Why?
2. Give one industrial use of dense flow and one of sparse flow.
3. Where in today's session did you meet the structure tensor again, and what does it tell you there?

## 12. What's new (2025–26)

> - **Learned flow and tracking of any point**: RAFT and SEA-RAFT for dense flow; **CoTracker 3**, TAPIR/BootsTAP and **TAPNext** track arbitrary
>   points through long videos and occlusions — the modern successors of the LK trails in Section 5.
> - **Event cameras** (neuromorphic sensors) report per-pixel brightness changes with microsecond latency — frame differencing in hardware — and are
>   entering high-speed inspection and robotics.
> - **Video foundation models** (e.g. SAM 2 for video segmentation, V-JEPA 2 for motion understanding) learn motion cues from huge unlabelled video
>   collections; Week 11B uses SAM 2.

## 13. References and further exploration
- Horn, B. K. P. & Schunck, B. G. (1981). *Determining optical flow.* Artificial Intelligence 17. [doi:10.1016/0004-3702(81)90024-2](https://doi.org/10.1016/0004-3702(81)90024-2)
- Lucas, B. D. & Kanade, T. (1981). *An iterative image registration technique with an application to stereo vision.* IJCAI. [ijcai.org/Proceedings/81-2/Papers/017.pdf](https://www.ijcai.org/Proceedings/81-2/Papers/017.pdf)
- Shi, J. & Tomasi, C. (1994). *Good features to track.* CVPR. [doi:10.1109/CVPR.1994.323794](https://doi.org/10.1109/CVPR.1994.323794)
- Bouguet, J.-Y. (2000). *Pyramidal implementation of the Lucas Kanade feature tracker.* Intel Corporation (the OpenCV implementation).
- Stauffer, C. & Grimson, W. (1999). *Adaptive background mixture models for real-time tracking.* CVPR. [doi:10.1109/CVPR.1999.784637](https://doi.org/10.1109/CVPR.1999.784637)
- Zivkovic, Z. (2004). *Improved adaptive Gaussian mixture model for background subtraction* (MOG2). ICPR. [doi:10.1109/ICPR.2004.1333992](https://doi.org/10.1109/ICPR.2004.1333992)
- Farnebäck, G. (2003). *Two-frame motion estimation based on polynomial expansion.* SCIA. [doi:10.1007/3-540-45103-X_50](https://doi.org/10.1007/3-540-45103-X_50)
- Kroeger, T. et al. (2016). *Fast optical flow using dense inverse search.* ECCV. [arXiv:1603.03590](https://arxiv.org/abs/1603.03590)
- Baker, S. et al. (2011). *A database and evaluation methodology for optical flow* (Middlebury). IJCV 92. [doi:10.1007/s11263-010-0390-2](https://doi.org/10.1007/s11263-010-0390-2)
- Teed, Z. & Deng, J. (2020). *RAFT: Recurrent all-pairs field transforms for optical flow.* ECCV. [arXiv:2003.12039](https://arxiv.org/abs/2003.12039)
- Karaev, N. et al. (2024). *CoTracker3: Simpler and better point tracking by pseudo-labelling real videos.* [arXiv:2410.11831](https://arxiv.org/abs/2410.11831)
- Szeliski (2022), Ch. 9 (motion estimation). [szeliski.org/Book](https://szeliski.org/Book/) · OpenCV tutorials: *Optical flow*, *Background subtraction*. [docs.opencv.org](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html)

**Data:** `vtest.avi`, `rubberwhale1.png`, `rubberwhale2.png` from the OpenCV sample data; conveyor video generated by `partsim.py`.

## 14. Key takeaways
- Change detection: frame differencing sees only edges; **background models** (median, MOG2, KNN) segment whole moving objects from a static camera.
- The **optical-flow constraint** $I_x u + I_y v + I_t = 0$ is one equation per pixel: the **aperture problem**.
- **Lucas–Kanade** solves it by least squares in a window (the structure tensor again) on a **pyramid**; dense methods (Farnebäck, DIS, RAFT) give a
  flow vector per pixel.
- Measure flow accuracy with the **end-point error**; convert flow to **mm/s** with the calibration and frame rate.

**Next session (10B):** tracking — giving each detected object an identity over time, with Kalman filters, the Hungarian algorithm and ByteTrack.